<a href="https://colab.research.google.com/github/ozenyilmaz/scRNA-seq-agentic-ai/blob/main/scRNA_seq_agentic_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# # Agentic AI Target Prioritization Platform — scRNA-seq Heart Failure Drug Target Study
#
# Bu notebook, mevcut scRNA-seq ön-işleme hattınızı (filtreleme, normalizasyon, HVG, PCA,
# UMAP, Leiden kümeleme) TAMAMEN BOZMADAN korur ve buna ek olarak:
#   - PubMed literatür harmanlama (Entrez E-utilities)
#   - ChEMBL hedef doğrulama / klinik faz haritalama
#   - Entropi tabanlı, dinamik ağırlıklandırmalı çok-kriterli (MCDA) hedef önceliklendirme
# katmanlarını ekler.

# %%
!pip install scanpy[leiden] python-igraph louvain matplotlib numpy pandas requests

# %%
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
import requests
from requests.adapters import HTTPAdapter, Retry

# Görsellerin hücre içinde canlı gösterilmesi için matplotlib ayarı
%matplotlib inline

# Scanpy genel ayarları
sc.settings.verbosity = 3  # 3: Detaylı loglama (Hangi adımın ne kadar sürdüğünü ve RAM harcadığını gösterir)
sc.settings.set_figure_params(dpi=100, facecolor='white', vector_friendly=True)
sc.logging.print_header()

# Sonuçların kaydedileceği klasör
RESULTS_DIR = './scrna_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
sc.settings.figdir = RESULTS_DIR

In [ ]:
# ## ADIM 1: Veri Yükleme (DEĞİŞTİRİLMEDİ)

# %%
print("--- ADIM 1: Veri Yükleniyor ---")
adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()

# --- KRİTİK KONTROL ADIMI ---
print("\n[KONTROL] Ham Veri Yapısı:")
print(f"- Hücre (Gözlem) Sayısı (n_obs): {adata.n_obs}")
print(f"- Gen Sayısı (n_vars): {adata.n_vars}")
print(f"- Veri Tipi (Sparse Matrix mi?): {type(adata.X)}")
print(f"- İlk 5 Hücre Barcode Örneği: {adata.obs_names[:5].tolist()}")
print(f"- İlk 5 Gen İsmi Örneği: {adata.var_names[:5].tolist()}")

In [ ]:
# ## ADIM 2: QC Metrikleri (DEĞİŞTİRİLMEDİ)

# %%
print("--- ADIM 2: QC Metrikleri Hesaplanıyor ---")
# İnsan genleri için "MT-", fare çalışsaydınız "mt-" olmalıydı
adata.var['mt'] = adata.var_names.str.startswith('MT-')
adata.var['ribo'] = adata.var_names.str.startswith(('RPS', 'RPL'))

# Metrikleri hesapla
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'ribo'], percent_top=None, log1p=False, inplace=True)

# --- KRİTİK KONTROL VE İSTATİSTİK ADIMI ---
print("\n[KONTROL] QC Metrik Özeti (İlk 5 Hücre):")
display(adata.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

print("\n[KONTROL] Veri Seti Genel Dağılım İstatistikleri:")
display(adata.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].describe())

# Görsel Kontroller (Colab hücresinde anlık belirecektir)
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], jitter=0.4, multi_panel=True, show=True)
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt', show=True)
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', show=True)


In [ ]:
# ## ADIM 3: Filtreleme

# %%
print("--- ADIM 3: Filtreleme Uygulanıyor ---")
original_cells = adata.n_obs
original_genes = adata.n_vars

# Filtreleri uygula
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs.n_genes_by_counts < 2500, :]
adata = adata[adata.obs.pct_counts_mt < 5, :].copy()

# --- KRİTİK KONTROL ADIMI (Sanity Check) ---
cell_loss = original_cells - adata.n_obs
gene_loss = original_genes - adata.n_vars
cell_loss_pct = (cell_loss / original_cells) * 100

print("\n[KONTROL] Filtreleme Sonrası Değişim:")
print(f"- Başlangıç Hücre Sayısı: {original_cells} -> Kalan Hücre: {adata.n_obs}")
print(f"- Elenen Hücre Sayısı: {cell_loss} (Kayıp Oranı: %{cell_loss_pct:.2f})")
print(f"- Başlangıç Gen Sayısı: {original_genes} -> Kalan Gen: {adata.n_vars}")

# Mantık Kontrolü (Sanity Check Alarmı)
if cell_loss_pct > 30:
    print("\n[UYARI/ALARM]: Hücrelerinizin %30'undan fazlasını kaybettiniz! Eşik değerleriniz (özellikle mitokondriyal %5 sınırı) bu veri seti için çok katı olabilir. Eşikleri gevşetmeyi düşünün.")
else:
    print("\n[BAŞARILI]: Hücre kaybı kabul edilebilir sınırlar içinde (%10-%20 civarı).")


In [ ]:
# ## ADIM 4: Normalizasyon ve Log-Transform (DEĞİŞTİRİLMEDİ)

# %%
print("--- ADIM 4: Normalizasyon ve Log-Transform ---")
# Ham sayımları ilerisi için (marker analizi) katmanda yedekle
adata.layers['counts'] = adata.X.copy()

# Hücre başına toplam transkript sayısını 10,000'e eşitle
sc.pp.normalize_total(adata, target_sum=1e4)
# Log(x+1) dönüşümü yap
sc.pp.log1p(adata)

# Downstream görselleştirmeler için log-normalize halini .raw alanında sakla
adata.raw = adata

# --- KRİTİK KONTROL ADIMI ---
print("\n[KONTROL] Normalizasyon Doğrulaması:")
row_sums = adata.X.sum(axis=1)
# Seyrek matris veya normal array olma durumuna göre kontrol
if isinstance(row_sums, np.matrix):
    row_sums = row_sums.A1

print(f"- İlk 5 hücrenin normalizasyon sonrası toplam transkript değerleri: {row_sums[:5]}")
print(f"- Tüm hücrelerin toplamlarının 10000'e eşitlik kontrolü (Varyans): {np.var(row_sums):.6f}")


In [ ]:
# ## ADIM 5 & 6: HVG Seçimi ve Ölçekleme (DEĞİŞTİRİLMEDİ)

# %%
print("--- ADIM 5 & 6: HVG Seçimi ve Ölçekleme ---")
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

# Görsel Kontrol: HVG Grafiği
sc.pl.highly_variable_genes(adata, show=True)

# --- KRİTİK KONTROL ADIMI ---
n_hvgs = adata.var.highly_variable.sum()
print(f"\n[KONTROL] Seçilen Yüksek Varyanslı Gen (HVG) Sayısı: {n_hvgs}")
if n_hvgs < 1000 or n_hvgs > 4000:
    print("[TAVSİYE]: HVG sayısı genellikle 2000-3000 arasında olmalıdır. Parametreleri optimize edebilirsiniz.")

# Sadece HVG'leri tutarak veriyi filtrele ve ölçekle
adata = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata, max_value=10)

print("\n[KONTROL] Ölçekleme Sonrası Matris İstatistikleri (İlk 5 Gen):")
print(f"- Genlerin Yeni Ortalaması (0'a yakın olmalı): {np.mean(adata.X, axis=0)[:5]}")
print(f"- Genlerin Yeni Varyansı (1'e yakın olmalı): {np.var(adata.X, axis=0)[:5]}")


In [ ]:
# ## ADIM 7: PCA (DEĞİŞTİRİLMEDİ)

# %%
print("--- ADIM 7: PCA Analizi ---")
sc.tl.pca(adata, svd_solver='arpack', n_comps=50)

# Görsel Kontrol: Dirsek (Elbow) Grafiği
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50, show=True)
sc.pl.pca(adata, color='total_counts', show=True)


In [ ]:
 ## ADIM 8 & 9: kNN Grafiği ve UMAP (DEĞİŞTİRİLMEDİ)

# %%
print("--- ADIM 8 & 9: kNN Grafiği ve UMAP ---")
# Yukarıdaki elbow grafiğine göre n_pcs=40 seçildi
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)

# --- KRİTİK KONTROL ADIMI ---
print("\n[KONTROL] Hücre Gömme (Embedding) Kontrolü:")
print(f"- UMAP Koordinat Matrisi Boyutu: {adata.obsm['X_umap'].shape}")

# UMAP Görsel Kontrolü (Hücreler henüz renklenmemiş, ham bulut halinde olmalı)
sc.pl.umap(adata, color=['total_counts'], show=True)


In [ ]:
# ## ADIM 10, 11 & 12: Kümeleme ve Marker Analizi (DEĞİŞTİRİLMEDİ)

# %%
print("--- ADIM 10, 11 & 12: Kümeleme ve Marker Analizi ---")
sc.tl.leiden(adata, resolution=0.5)

print("\n[KONTROL] Küme Popülasyon Dağılımı:")
print(adata.obs['leiden'].value_counts())

# Kümeleri UMAP üzerinde göster
sc.pl.umap(adata, color=['leiden'], show=True)

# Wilcoxon test ile marker genleri bul
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')

# İlk 5 marker geni tablo olarak kontrol et
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
top_markers = pd.DataFrame({group: result['names'][group][:5] for group in groups})
print("\n[KONTROL] Her Küme İçin En Belirgin (Top 5) Marker Genler:")
display(top_markers)

# Biyolojik Doğrulama Kontrolü (Dotplot)
marker_genes = ['IL7R', 'CD4', 'CD8A', 'MS4A1', 'CD79A', 'GNLY', 'NKG7', 'CD14', 'LYZ']
available_markers = [g for g in marker_genes if g in adata.raw.var_names]
sc.pl.dotplot(adata, available_markers, groupby='leiden', show=True)

In [ ]:
# ## ADIM 13: AGENTIC AI KNOWLEDGE-HARMONIZATION LAYER
#
# Bu bölüm yeni eklenen bilgi-harmanlama ve önceliklendirme katmanıdır.
# Yukarıdaki tüm preprocessing / clustering adımları korunmuştur; bu katman
# sadece `rank_genes_groups` çıktısını girdi olarak alır.


# ### 13.1 — API İstemcileri (PubMed E-utilities & ChEMBL)

# %%
def get_requests_session():
    """Geri-deneme (retry) mantığı içeren paylaşılan bir HTTP oturumu döndürür."""
    session = requests.Session()
    retries = Retry(total=3, backoff_factor=0.5, status_forcelist=[429, 500, 502, 503, 504])
    session.mount('https://', HTTPAdapter(max_retries=retries))
    return session


SESSION = get_requests_session()

PUBMED_ESEARCH = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
CHEMBL_TARGET = "https://www.ebi.ac.uk/chembl/api/data/target.json"
CHEMBL_MECHANISM = "https://www.ebi.ac.uk/chembl/api/data/mechanism.json"
CHEMBL_MOLECULE = "https://www.ebi.ac.uk/chembl/api/data/molecule.json"

# Bu çalışma "Heart Failure" inceleme makalesi için olduğundan, literatür
# sorguları bu hastalık bağlamına göre filtrelenir. Gerekirse değiştirin.
DISEASE_CONTEXT = "heart failure"


def fetch_pubmed_count(gene_symbol, disease_context=DISEASE_CONTEXT, session=SESSION, sleep=0.34):
    """
    Belirtilen gen sembolü ile hastalık bağlamının (örn. 'heart failure')
    PubMed Title/Abstract alanlarında birlikte geçtiği makale sayısını döndürür.
    NCBI E-utilities, anahtarsız erişimde ~3 istek/saniye sınırına sahiptir;
    bu yüzden her çağrı sonrası kısa bir bekleme uygulanır.
    """
    term = f"{gene_symbol}[Title/Abstract] AND {disease_context}[Title/Abstract]"
    params = {"db": "pubmed", "term": term, "retmode": "json", "retmax": 0}
    try:
        r = session.get(PUBMED_ESEARCH, params=params, timeout=15)
        r.raise_for_status()
        data = r.json()
        count = int(data.get("esearchresult", {}).get("count", 0))
    except Exception as e:
        print(f"  [PubMed UYARI] {gene_symbol}: {e}")
        count = np.nan
    time.sleep(sleep)
    return count


def fetch_chembl_max_phase(gene_symbol, session=SESSION, sleep=0.2):
    """
    ChEMBL ilişkisel şemasını resmi yoldan dolaşır:
        target (Homo sapiens, tam eşleşen sözdizimi/synonym, büyük/küçük harf duyarsız)
          -> mechanism (target_chembl_id -> molecule_chembl_id)
          -> molecule (molecule_chembl_id -> max_phase)

    Döndürülen değer, bu hedefe etki ettiği belgelenmiş herhangi bir bileşiğin
    ulaştığı en yüksek klinik faz (0-4) değeridir. Hiçbir doğrulanmış hedef/
    mekanizma bulunamazsa 0 döner (yanlış-pozitif çapraz-tür eşleşmesini önlemek
    için organism alanı 'Homo sapiens' ile katı şekilde kısıtlanmıştır).
    """
    try:
        # --- 1) Hedef pasaport ID'si: tam eşleşen, büyük/küçük harf duyarsız synonym, sadece insan ---
        params = {
            "target_synonym__iexact": gene_symbol,
            "organism__iexact": "Homo sapiens",
            "format": "json",
        }
        r = session.get(CHEMBL_TARGET, params=params, timeout=15)
        r.raise_for_status()
        targets = r.json().get("targets", [])
        time.sleep(sleep)
        if not targets:
            return 0  # doğrulanmış insan hedefi yok

        target_ids = [t["target_chembl_id"] for t in targets if "target_chembl_id" in t]

        # --- 2) mechanism endpoint: hedefe bağlı molecule_chembl_id'leri topla ---
        molecule_ids = set()
        for tid in target_ids:
            r = session.get(CHEMBL_MECHANISM, params={"target_chembl_id": tid, "format": "json"}, timeout=15)
            r.raise_for_status()
            for m in r.json().get("mechanisms", []):
                mol_id = m.get("molecule_chembl_id")
                if mol_id:
                    molecule_ids.add(mol_id)
            time.sleep(sleep)

        if not molecule_ids:
            return 0  # doğrulanmış hedef var ama belgelenmiş bir mekanizma yok

        # --- 3) molecule endpoint: gerçek max_phase değerini oku ---

        max_phase = 0
        molecule_ids = list(molecule_ids)
        CHUNK = 25
        for i in range(0, len(molecule_ids), CHUNK):
            chunk = molecule_ids[i:i + CHUNK]
            r = session.get(
                CHEMBL_MOLECULE,
                params={"molecule_chembl_id__in": ",".join(chunk), "format": "json"},
                timeout=15,
            )
            r.raise_for_status()
            for mol in r.json().get("molecules", []):
                raw_phase = mol.get("max_phase")

                if raw_phase is not None:
                    try:
                        # Önce float'a çevirerek '4.0' -> 4.0 yapılır, sonra int'e dökülerek 4 elde edilir.
                        phase = int(float(raw_phase))
                        if phase > max_phase:
                            max_phase = phase
                    except (ValueError, TypeError):
                        continue
            time.sleep(sleep)

        return max_phase

    except Exception as e:
        print(f"  [ChEMBL UYARI] {gene_symbol}: {e}")
        return np.nan



# ### 13.2 — Non-Linear Skorlama Fonksiyonları
#
# - `literature_saturation_score`: log1p(PubMed sayısı) üzerinde, aday
#   havuzunun medyanına ve IQR'sine göre dinamik olarak merkezlenmiş/eğimi
#   ayarlanmış bir lojistik (sigmoid) doygunluk eğrisi.
# - `chembl_phase_score`: max_phase (0-4) üzerinde, Faz 2'nin biraz üzerinde
#   merkezlenmiş, Faz 4'ü (onaylı ilaç) güçlü biçimde ama doygun şekilde
#   ödüllendiren lojistik dönüşüm.

# %%
def literature_saturation_score(counts):
    """
    PubMed sayım dizisini [0, 1] aralığında bir 'literatür doygunluk skoruna'
    dönüştürür. Sabit bir eşik kullanmak yerine, sigmoidin orta noktası
    log1p(sayı) dağılımının MEDYANINA, eğimi ise IQR'nin tersine bağlanır.
    Böylece eğri, taranan her gen kümesine kendiliğinden uyum sağlar:
    10 ile 500 hit arasındaki fark, 500 ile 3000 hit arasındaki farktan
    daha fazla ayırt edici güce sahip olur (doygunluk bölgesinde sıkışma).
    """
    counts = np.asarray(counts, dtype=float)
    x = np.log1p(np.nan_to_num(counts, nan=0.0))

    median = np.median(x)
    q75, q25 = np.percentile(x, 75), np.percentile(x, 25)
    iqr = q75 - q25
    iqr = iqr if iqr > 1e-6 else 1.0

    k = 4.0 / iqr  # eğim, dağılımın yayılımıyla ters orantılı -> tutarlı ayırt edicilik
    score = 1.0 / (1.0 + np.exp(-k * (x - median)))
    return score


def chembl_phase_score(max_phases):
    """
    ChEMBL max_phase (0-4) değerlerini [0, 1] aralığında bir 'klinik doğrulama
    skoruna' dönüştürür. Lojistik eğri Faz ~2.2 civarında merkezlenir:
      - Faz 0 -> ~0  (hedef olarak doğrulanmamış / ilaç yok)
      - Faz 1-3 -> kademeli, belirgin artış (aktif araştırma hedefi)
      - Faz 4 -> ~1  (FDA onaylı, en yüksek klinik doğrulama)
    Doğrusal (phase/4) bir haritalamanın aksine, Faz 3->4 geçişindeki
    niteliksel sıçramayı (deneysel -> onaylı) doğru şekilde vurgular.
    """
    p = np.asarray(max_phases, dtype=float)
    p = np.nan_to_num(p, nan=0.0)

    raw = 1.0 / (1.0 + np.exp(-1.8 * (p - 2.2)))
    lo = 1.0 / (1.0 + np.exp(-1.8 * (0.0 - 2.2)))
    hi = 1.0 / (1.0 + np.exp(-1.8 * (4.0 - 2.2)))
    score = (raw - lo) / (hi - lo)
    return np.clip(score, 0.0, 1.0)



# ### 13.3 — Entropi Tabanlı Dinamik Çok-Kriterli (MCDA) Ağırlıklandırma
#
# Hardcoded ağırlıklar yerine, her kriterin (LogFC, literatür skoru, ChEMBL
# skoru) aday havuzu içindeki DAĞILIMINA (Shannon entropisi) bağlı, objektif
# bir ağırlık hesaplanır. Daha ayırt edici (yüksek varyanslı) kriterler
# otomatik olarak daha yüksek ağırlık alır. LogFC kanalına, transkriptomik
# sadakatin birincilliğini garanti eden bir taban ağırlık (>= min_logfc_weight)
# uygulanır; kalan ağırlık bütçesi, entropiye dayalı bilgi-değerleriyle
# orantılı olarak literatür ve ChEMBL kriterleri arasında paylaştırılır.

# %%
def entropy_weights(matrix, min_logfc_weight=0.5):
    """
    matrix: (n_candidates, n_criteria) şeklinde, sütun 0 = LogFC (normalize),
            sütun 1..k = bilgi katmanı skorları (normalize, [0,1]).
    Dönen değer: her kritere ait, toplamı 1 olan ağırlık vektörü.
    """
    eps = 1e-12
    m = np.asarray(matrix, dtype=float)

    # Her sütunu [eps, 1] aralığına min-max normalize et
    col_min = m.min(axis=0)
    col_max = m.max(axis=0)
    denom = np.where((col_max - col_min) < eps, 1.0, col_max - col_min)
    norm = (m - col_min) / denom
    norm = np.clip(norm, eps, 1.0)

    # Her kriter için olasılık dağılımı
    P = norm / norm.sum(axis=0, keepdims=True)
    n = m.shape[0]
    k = 1.0 / np.log(n) if n > 1 else 1.0
    entropy = -k * (P * np.log(P)).sum(axis=0)

    diversity = np.clip(1.0 - entropy, eps, None)  # ayırt edicilik derecesi
    raw_weights = diversity / diversity.sum()

    # LogFC (sütun 0) için biyolojik-sadakat taban ağırlığı
    w_logfc = max(raw_weights[0], min_logfc_weight)
    remaining = 1.0 - w_logfc

    other_raw = raw_weights[1:]
    other_sum = other_raw.sum()
    if other_sum < eps:
        other_weights = np.full_like(other_raw, remaining / len(other_raw))
    else:
        other_weights = remaining * (other_raw / other_sum)

    return np.concatenate([[w_logfc], other_weights])



# ### 13.4 — Marker Döngüsünün Agentic Hale Getirilmesi: Master Önceliklendirme Tablosu
#
# Her küme için en üst sıradaki marker genler alınır, her BENZERSİZ gen için
# PubMed ve ChEMBL API'leri bir kez sorgulanır (önbellekleme ile), ve tüm
# kriterler birleşik bir MCDA skoruna indirgenir.

# %%
print("--- ADIM 13: Agentic Bilgi Harmanlama ve Hedef Önceliklendirme ---")

TOP_N_MARKERS = 15  # küme başına değerlendirilecek en üst marker sayısı

result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names

candidate_rows = []
for group in groups:
    names = result['names'][group][:TOP_N_MARKERS]
    lfc = result['logfoldchanges'][group][:TOP_N_MARKERS]
    pvals_adj = result['pvals_adj'][group][:TOP_N_MARKERS]
    for gene, fc, p in zip(names, lfc, pvals_adj):
        candidate_rows.append({
            "cluster": group,
            "gene": gene,
            "logFC": float(fc),
            "pval_adj": float(p),
        })

master_df = pd.DataFrame(candidate_rows)
print(f"[BİLGİ] Toplam aday gen-küme çifti: {len(master_df)}")

unique_genes = master_df['gene'].unique().tolist()
print(f"[BİLGİ] Benzersiz gen sayısı (API sorgulanacak): {len(unique_genes)}")

# --- Önbellekli API harvesting: her gen sadece bir kez sorgulanır ---
pubmed_cache = {}
chembl_cache = {}

for i, gene in enumerate(unique_genes):
    print(f"  [{i + 1}/{len(unique_genes)}] Sorgulanıyor: {gene}")
    pubmed_cache[gene] = fetch_pubmed_count(gene)
    chembl_cache[gene] = fetch_chembl_max_phase(gene)

master_df['pubmed_count'] = master_df['gene'].map(pubmed_cache)
master_df['chembl_max_phase'] = master_df['gene'].map(chembl_cache)

# --- KRİTİK KONTROL ADIMI ---
print("\n[KONTROL] API Harvesting Sonrası Örnek Satırlar:")
display(master_df.head(10))

n_missing_pubmed = master_df['pubmed_count'].isna().sum()
n_missing_chembl = master_df['chembl_max_phase'].isna().sum()
if n_missing_pubmed or n_missing_chembl:
    print(f"[UYARI]: {n_missing_pubmed} satırda PubMed, {n_missing_chembl} satırda ChEMBL "
          f"sonucu alınamadı (ağ hatası olabilir). Bu değerler 0 olarak ele alınacak.")


# ### 13.5 — Non-Lineer Dönüşümler, Dinamik Ağırlıklandırma ve Birleşik Skor

# %%
# Non-lineer doygunluk / faz dönüşümleri
master_df['literature_score'] = literature_saturation_score(master_df['pubmed_count'].values)
master_df['chembl_score'] = chembl_phase_score(master_df['chembl_max_phase'].values)

# LogFC'yi global olarak [0, 1]'e min-max normalize et (transkriptomik kanal)
lfc = master_df['logFC'].values.astype(float)
lfc_min, lfc_max = lfc.min(), lfc.max()
master_df['logfc_norm'] = (lfc - lfc_min) / (lfc_max - lfc_min + 1e-12)

# Entropi tabanlı dinamik ağırlıklar (LogFC için taban ağırlık = 0.5)
criteria_matrix = master_df[['logfc_norm', 'literature_score', 'chembl_score']].values
weights = entropy_weights(criteria_matrix, min_logfc_weight=0.5)

print("\n[KONTROL] Dinamik Olarak Hesaplanan MCDA Ağırlıkları:")
print(f"  - LogFC (Transkriptomik Sadakat):  {weights[0]:.3f}")
print(f"  - Literatür Doygunluk Skoru:       {weights[1]:.3f}")
print(f"  - ChEMBL Klinik Faz Skoru:         {weights[2]:.3f}")
print(f"  - Toplam: {weights.sum():.3f}")

master_df['priority_score'] = criteria_matrix @ weights
master_df = master_df.sort_values(
    ['cluster', 'priority_score'], ascending=[True, False]
).reset_index(drop=True)

# --- Sonuçları Kalıcı Kaydetme ---
output_csv = f'{RESULTS_DIR}/target_prioritization_master.csv'
master_df.to_csv(output_csv, index=False)
print(f"\n[BAŞARILI] Birleşik hedef önceliklendirme tablosu kaydedildi: {output_csv}")


# ### 13.6 — Küme Bazlı Özet: İlk 3 Öncelikli Hedef

# %%
print("\n[ÖZET] Her Küme İçin İlk 3 Öncelikli Hedef:")
for group in groups:
    print(f"\n--- Küme {group} ---")
    top3 = master_df[master_df['cluster'] == group].head(3)
    display(top3[['gene', 'logFC', 'pubmed_count', 'chembl_max_phase', 'priority_score']])


In [ ]:
# ## ADIM 14: Nesneyi Kalıcı Kaydetme (DEĞİŞTİRİLMEDİ)

# %%
output_h5ad = f'{RESULTS_DIR}/pbmc3k_processed_colab.h5ad'
adata.write(output_h5ad)
print(f"\n[BAŞARILI]: Tüm analiz tamamlandı ve nesne '{output_h5ad}' olarak kaydedildi!")
print(f"[BAŞARILI]: Hedef önceliklendirme tablosu '{output_csv}' olarak kaydedildi!")